# ArgLLM baseline (DF-QuAD, re-implemented prompts) — Gemma 4 31B on αNLI (500 claims, sample hash 24b904e78d)

Depth-one ArgLLM with the option to decline an argument, strengths given as a number between 0 and 1 (unreadable answers set to 0.5), temperature 0.7, top-p 0.95. Results: 0.5 Base 59.0, Est. Base 67.6. Run on Kaggle; the API key is read from Kaggle secrets.

In [2]:
# ===== DF-QuAD — PART 1: setup + sample + prompts (Gemma + αNLI) =====
!pip install -q openai datasets scikit-learn

import os, re, json, time, pickle, hashlib, sys, zipfile, io, base64, urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np, pandas as pd
from openai import OpenAI
from kaggle_secrets import UserSecretsClient
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix)
from IPython.display import HTML, display

LABEL      = "Gemma 4 31B"
MODEL_NAME = "google/gemma-4-31b-it"
DATASET    = "aNLI"
EXPECTED   = "24b904e78d"
RUN_NAME   = "dfquad_gemma_anli"
WORKERS    = 4
client = OpenAI(api_key=UserSecretsClient().get_secret("OPENROUTER_API_KEY"),
                base_url="https://openrouter.ai/api/v1", timeout=180)

WORK_DIR = Path("/kaggle/working") / RUN_NAME
WORK_DIR.mkdir(parents=True, exist_ok=True)
CKPT = WORK_DIR / f"{RUN_NAME}_ckpt.pkl"
SEED = 42


# ---- SAME αNLI sample as our method (dev split, seed 42) ----
def sentence(s):
    s = str(s).strip()
    return s if s.endswith((".", "!", "?")) else s + "."


try:
    url = "https://storage.googleapis.com/ai2-mosaic/public/abductive-commonsense-reasoning-iclr2020/anli.zip"
    zf = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url, timeout=120).read()))
    names = zf.namelist()
    jl = [n for n in names if n.endswith("dev.jsonl")][0]
    lb = [n for n in names if n.endswith("dev-labels.lst")][0]
    rows = [json.loads(l) for l in zf.read(jl).decode().splitlines() if l.strip()]
    raw = pd.DataFrame(rows)
    raw["label"] = [int(l) for l in zf.read(lb).decode().split()]
    raw = raw[["obs1", "obs2", "hyp1", "hyp2", "label"]]
    print("loaded αNLI dev from the official zip")
except Exception as e:
    print("official zip failed -> Hugging Face fallback:", str(e)[:100])
    from datasets import load_dataset
    raw = load_dataset("allenai/art", split="validation").to_pandas().rename(
        columns={"observation_1": "obs1", "observation_2": "obs2",
                 "hypothesis_1": "hyp1", "hypothesis_2": "hyp2"})[["obs1", "obs2", "hyp1", "hyp2", "label"]]

raw = raw[raw["label"].isin([1, 2])].copy()
for c in ["obs1", "obs2", "hyp1", "hyp2"]:
    raw[c] = raw[c].astype(str).str.strip()
raw["item_id"] = [hashlib.md5("|".join(r).encode()).hexdigest()[:12]
                  for r in raw[["obs1", "obs2", "hyp1", "hyp2"]].values.tolist()]
raw = raw.drop_duplicates("item_id").sort_values("item_id").reset_index(drop=True)
items = raw.sample(n=500, random_state=SEED).reset_index(drop=True)
gold_arr = np.array([1] * 250 + [0] * 250)[np.random.RandomState(SEED).permutation(500)]
claims = []
for (_, r), g in zip(items.iterrows(), gold_arr):
    right = r["hyp1"] if r["label"] == 1 else r["hyp2"]
    wrong = r["hyp2"] if r["label"] == 1 else r["hyp1"]
    claims.append(f"{sentence(r['obs1'])} {sentence(r['obs2'])} This is because: "
                  f"{sentence(right if g == 1 else wrong)}")
gold = gold_arr.astype(int).tolist()
SAMPLE_HASH = hashlib.md5("|".join(f"{s}:{g}" for s, g in zip(items["item_id"], gold)).encode()).hexdigest()[:10]
print(f"{LABEL} | {DATASET} | claims: {len(claims)} | true: {sum(gold)} | SAMPLE HASH: {SAMPLE_HASH}")
assert SAMPLE_HASH == EXPECTED, f"Different sample! Expected {EXPECTED}."

# ---- DF-QuAD library (Uncertainpy, from the ArgLLM repo) ----
repo = "/kaggle/working/argumentative-llms"
if not Path(repo).exists():
    os.system("cd /kaggle/working && git clone -q https://github.com/CLArg-group/argumentative-llms.git")
sys.path.insert(0, repo)
try:
    import Uncertainpy.src.uncertainpy.gradual as grad
    agg_f = grad.semantics.modular.ProductAggregation()
    inf_f = grad.semantics.modular.LinearInfluence(conservativeness=1)
    HAVE_UNCERTAINPY = True
except Exception as e:
    print("Uncertainpy not importable, using built-in DF-QuAD only:", e)
    HAVE_UNCERTAINPY = False


def llm_raw(prompt, max_tokens=200, retries=8):
    last = None
    for a in range(1, retries + 1):
        try:
            r = client.chat.completions.create(
                model=MODEL_NAME, messages=[{"role": "user", "content": prompt}],
                temperature=0.7, top_p=0.95, max_tokens=max_tokens)
            t = r.choices[0].message.content
            if t:
                return t.strip()
            raise ValueError("empty")
        except Exception as e:
            last = e; msg = str(e)
            if getattr(e, "status_code", None) == 402 or "Error code: 402" in msg:
                raise RuntimeError(f"no API credit left: {msg[:200]}")
            busy = any(c in msg for c in ["429", "502", "503", "504"]) or "rate" in msg.lower()
            time.sleep(20 * a if busy else 3 * a)
    raise RuntimeError(f"failed: {last}")


def am_prompt(claim, kind):
    verb = "support" if kind == "supporting" else "attack"
    cond = ("there is a non-zero probability that the claim is true" if kind == "supporting"
            else "there is a non-zero probability that the claim is false")
    return (f"Please provide a single short argument {kind} the following claim. "
            f"Construct the argument so it refers to the truthfulness of the claim. "
            f"Only provide an argument if you think there is a valid and convincing {verb} "
            f"for this claim ({cond}), otherwise return: N/A.\n"
            f"Claim: {claim}\nArgument:")


def ue_arg_prompt(claim, arg, kind):
    return (f"Consider the following claim and an argument {kind} it.\n"
            f"Claim: {claim}\nArgument: {arg}\n"
            f"How strong is this argument, on a scale from 0 (very weak) to 1 (very strong)? "
            f"Answer with a single number between 0 and 1 and nothing else.")


def ue_topic_prompt(claim):
    return (f"Claim: {claim}\n"
            f"How likely is this claim to be true, on a scale from 0 (certainly false) to 1 (certainly true)? "
            f"Answer with a single number between 0 and 1 and nothing else.")


def gen_argument(claim, kind):
    t = llm_raw(am_prompt(claim, kind), max_tokens=120)
    t = t.replace("Argument:", "").strip().strip('"')
    return "N/A" if (not t or "N/A" in t[:10].upper()) else t


def get_lik(prompt):
    t = llm_raw(prompt, max_tokens=10)
    m = re.search(r"(?<![\d.])(0(?:\.\d+)?|1(?:\.0+)?|\.\d+)(?![\d.])", t)
    return (float(m.group(1)), True) if m else (0.5, False)   # default 0.5, as in ArgLLM


print("PART 1 ready -> now run PART 2")

loaded αNLI dev from the official zip
Gemma 4 31B | aNLI | claims: 500 | true: 250 | SAMPLE HASH: 24b904e78d
PART 1 ready -> now run PART 2


In [3]:
# ===== DF-QuAD — PART 2: strength, run, results, backup =====
def dfquad_manual(tau, sup, att):
    vs = sup if sup is not None else 0.0
    va = att if att is not None else 0.0
    if va >= vs:
        return tau - tau * (va - vs)
    return tau + (1 - tau) * (vs - va)


def dfquad_strength(v, est):
    tau = v["topic_base"] if est else 0.5
    sup = None if v["sup"] == "N/A" else float(v["sup_base"])
    att = None if v["att"] == "N/A" else float(v["att_base"])
    s = dfquad_manual(float(tau), sup, att)
    if HAVE_UNCERTAINPY:
        bag = grad.BAG()
        c = grad.Argument("db0", v["claim"], float(tau)); bag.arguments["db0"] = c
        if sup is not None: bag.add_support(grad.Argument("S1", "s", sup), c)
        if att is not None: bag.add_attack(grad.Argument("A1", "a", att), c)
        grad.algorithms.computeStrengthValues(bag, agg_f, inf_f)
        s_lib = float(bag.arguments["db0"].strength)
        if abs(s_lib - s) > 1e-6:
            print(f"mismatch {s:.4f} vs {s_lib:.4f}")
        s = s_lib
    return s


def run_one(i):
    c = claims[i]
    v = {"claim": c, "gold": gold[i], "unreadable": 0}
    v["sup"] = gen_argument(c, "supporting")
    v["att"] = gen_argument(c, "attacking")
    v["sup_base"] = v["att_base"] = None
    if v["sup"] != "N/A":
        v["sup_base"], ok = get_lik(ue_arg_prompt(c, v["sup"], "supporting")); v["unreadable"] += not ok
    if v["att"] != "N/A":
        v["att_base"], ok = get_lik(ue_arg_prompt(c, v["att"], "attacking")); v["unreadable"] += not ok
    v["topic_base"], ok = get_lik(ue_topic_prompt(c)); v["unreadable"] += not ok
    v["s_05"] = dfquad_strength(v, False)
    v["s_est"] = dfquad_strength(v, True)
    return v


def safe_one(i):
    try:
        return i, run_one(i), None
    except Exception as e:
        return i, None, str(e)[:120]


def load_ck():
    return pickle.load(open(CKPT, "rb")) if CKPT.exists() else {}


def save_ck(d):
    tmp = CKPT.with_suffix(".tmp"); pickle.dump(d, open(tmp, "wb")); tmp.replace(CKPT)


def report(res):
    idx = sorted(res); y = [res[i]["gold"] for i in idx]; rows = []
    for name, col in [("DF-QuAD 0.5 Base", "s_05"), ("DF-QuAD Est. Base", "s_est")]:
        p = [int(res[i][col] > 0.5) for i in idx]
        tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
        rows.append({"method": name, "n": len(idx),
                     "acc": round(accuracy_score(y, p) * 100, 1),
                     "prec": round(precision_score(y, p, zero_division=0) * 100, 1),
                     "rec": round(recall_score(y, p, zero_division=0) * 100, 1),
                     "f1": round(f1_score(y, p, zero_division=0) * 100, 1),
                     "TN": tn, "FP": fp, "FN": fn, "TP": tp})
    return pd.DataFrame(rows)


# ---------- run (resumable) ----------
for attempt in range(3):
    d = load_ck()
    todo = [i for i in range(len(claims)) if i not in d]
    print(f"[pass {attempt + 1}] have {len(d)}/{len(claims)}, running {len(todo)}", flush=True)
    if not todo:
        break
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        for f in as_completed([pool.submit(safe_one, i) for i in todo]):
            i, v, err = f.result()
            if err:
                print(f"  error at {i}: {err}", flush=True); continue
            d[i] = v; save_ck(d)
            if len(d) % 50 == 0:
                print(f"  {len(d)}/{len(claims)} | {(time.time() - t0) / 60:.0f} min", flush=True)
    time.sleep(10)

d = load_ck()
table = report(d)
pd.set_option("display.width", 220)
print(f"\n=== FINAL: {LABEL} | {DATASET} | DF-QuAD (no facts) | sample {SAMPLE_HASH} ({len(d)}/{len(claims)}) ===")
print(table.to_string(index=False))
print("N/A sup:", sum(v["sup"] == "N/A" for v in d.values()),
      "| N/A att:", sum(v["att"] == "N/A" for v in d.values()))
print("unreadable strength scores (default 0.5):", sum(v["unreadable"] for v in d.values()))

# ---------- backup: embedded Download button ----------
csv = WORK_DIR / f"{RUN_NAME}_results.csv"
table.to_csv(csv, index=False)
pkl = WORK_DIR / f"{RUN_NAME}_{len(d)}.pkl"
pickle.dump({"results": d, "sample_hash": SAMPLE_HASH}, open(pkl, "wb"))
zp = WORK_DIR / f"{RUN_NAME}_backup.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(pkl, arcname=pkl.name); z.write(csv, arcname=csv.name)
b64 = base64.b64encode(open(zp, "rb").read()).decode()
display(HTML(f'<a download="{zp.name}" href="data:application/zip;base64,{b64}" '
             f'style="display:inline-block;padding:12px 20px;background:#2e7d32;color:white;'
             f'border-radius:6px;text-decoration:none;font-weight:bold;">⬇ Download {zp.name}</a>'))

[pass 1] have 0/500, running 500
  50/500 | 1 min
  100/500 | 2 min
  150/500 | 3 min
  200/500 | 4 min
  250/500 | 5 min
  300/500 | 6 min
  350/500 | 7 min
  400/500 | 7 min
  450/500 | 8 min
  500/500 | 9 min
[pass 2] have 500/500, running 0

=== FINAL: Gemma 4 31B | aNLI | DF-QuAD (no facts) | sample 24b904e78d (500/500) ===
           method   n  acc  prec  rec   f1  TN  FP  FN  TP
 DF-QuAD 0.5 Base 500 59.0  72.3 29.2 41.6 222  28 177  73
DF-QuAD Est. Base 500 67.6  77.8 49.2 60.3 215  35 127 123
N/A sup: 86 | N/A att: 0
unreadable strength scores (default 0.5): 0
